# Factura → tabla recortada → OCR → items

```
imagen original
   ↓  (1) LLM de visión  →  bounding box de la tabla de items
   ↓  (2) enderezado     →  corrige la inclinación de la tabla
imagen recortada + zoom
   ↓  (3) PaddleOCR      →  texto + coordenadas
   ↓  (4) agrupado por y →  filas
   ↓  (5) cortes de columna →  items nombrados + validación aritmética
```

**Este notebook no tiene lógica.** Todo vive en los `.py` del repo, un módulo por
paso: `vision.py` (dónde está la tabla), `imagen.py` (orientación, enderezado,
recorte), `ocr.py` (PaddleOCR), `filas.py` (cajas → filas), `esquema.py` (dónde
empieza cada columna) y `encabezado.py` (cómo se llama cada columna).
`pipeline.py` es la fachada: orquesta y re-exporta, y es lo único que este
notebook toca.
Para cambiar el comportamiento se edita el archivo que corresponda, se pushea, y
acá alcanza con volver a correr la celda de *setup*.

> Colab: **Entorno de ejecución → Cambiar tipo → GPU T4** (opcional, más rápido).


## 0. Setup — traer el repo e instalar

Corré esta celda cada vez que haya cambios en `pipeline.py`. El `git pull` y el
`reload` hacen que los cambios entren sin reiniciar el entorno.


In [ ]:
REPO_URL = "https://github.com/FedericoRojo/imageProccesing.git"
CARPETA = "tesis-facturas"

import subprocess, sys
from pathlib import Path

# Repo privado -> cargá un token de GitHub en los secrets de Colab como
# GITHUB_TOKEN (fine-grained, permiso Contents: read).
# Si hacés el repo público, no hace falta ningún token.
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL


def _correr(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    salida = (r.stdout + r.stderr)
    if tok:
        salida = salida.replace(tok, "***")   # que el token no quede en el output
    if r.returncode:
        raise RuntimeError(salida)
    return salida


if Path(CARPETA).exists():
    print(_correr(["git", "-C", CARPETA, "pull", "--ff-only"]))
else:
    _correr(["git", "clone", "--depth", "1", url, CARPETA])
    print("clonado")

sys.path.insert(0, str(Path(CARPETA).resolve()))
print("repo listo")

In [ ]:
# Instalación (sólo la primera vez del entorno; después es instantánea)
!pip install -q -r {CARPETA}/requirements.txt
print("dependencias listas")

In [ ]:
# Importar / recargar los módulos. Correr después de cada git pull,
# y también si Colab reinició el entorno (se pierde el sys.path del setup).
import sys
from pathlib import Path

CARPETA = "tesis-facturas"
_ruta = str(Path(CARPETA).resolve())
if not Path(_ruta).exists():
    raise RuntimeError(f"No encuentro {CARPETA}/. Corré primero la celda de setup.")
if _ruta not in sys.path:
    sys.path.insert(0, _ruta)

import importlib

# En orden de dependencia, y `pipeline` SIEMPRE último: es una fachada que hace
# `from vision import ...`, así que recargarlo antes que los módulos que trae
# dejaría sus nombres apuntando al código viejo.
# Y recargar sólo `pipeline` no alcanza: un cambio en `esquema.py` que trajo el
# git pull no se vería, porque el módulo ya cargado queda en cache.
import texto, encabezado, esquema, vision, imagen, ocr, filas, diagnostico, pipeline
for _m in (texto, encabezado, esquema, vision, imagen, ocr, filas, diagnostico, pipeline):
    importlib.reload(_m)

print("pipeline listo:", pipeline.__all__)


## 1. Entrada — subir la factura

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()              # .png / .jpg / .jpeg
IMG = Path(list(uploaded.keys())[0])
print("Archivo:", IMG)

# Si el archivo ya está en /content:
# IMG = Path("/content/factura2.jpeg")

## 2. Imagen → LLM → tabla recortada

Necesita `NVIDIA_API_KEY` en los secrets de Colab (ícono de la llave, panel izquierdo).


In [ ]:
from PIL import Image
from IPython.display import display

IMG_TABLA = pipeline.preparar_tabla(IMG)      # detecta el bbox y recorta
print("Tabla recortada:", IMG_TABLA)         # queda guardada en recortes/

display(Image.open(IMG_TABLA))                # revisá el recorte antes de seguir

Los recortes se guardan en `recortes/` (panel de archivos de Colab, ícono de la carpeta a la izquierda). Para bajarlos a tu máquina — un PNG si hay uno solo, un `.zip` si hay varios:

In [ ]:
pipeline.descargar_recortes()

Si el recorte quedó mal:

- corta filas → `pipeline.preparar_tabla(IMG, margen=0.05)`
- texto chico → `pipeline.preparar_tabla(IMG, zoom=3)`
- el LLM no encuentra nada → devuelve la imagen original y el pipeline sigue igual

El recorte sale **enderezado**: la tabla se rota para dejar los renglones
horizontales antes del zoom. Las facturas del corpus vienen inclinadas entre
0.14° y 0.53°, que parece nada y no lo es — sobre el ancho de la tabla son ~13 px
de deriva contra un paso entre renglones de 27 px, y eso alcanzaba para que el
agrupado encadenara una fila con la siguiente.

Para comparar contra la línea de base: `pipeline.preparar_tabla(IMG, enderezado=False)`.


## 3. Tabla recortada → OCR

In [ ]:
res = pipeline.ocr_tabla(IMG_TABLA)
print(len(res["rec_texts"]), "cajas de texto")

# Lo que peor leyó, primero — útil para detectar dónde falla
pipeline.texto_con_score(res).head(15)

In [ ]:
# Imagen anotada con las cajas detectadas
import glob
from IPython.display import Image as IPyImage, display

for p in glob.glob("output/*.jpg") + glob.glob("output/*.png"):
    display(IPyImage(p))

## 4. OCR → filas

`agrupar_filas` hace dos cosas antes de agrupar:

1. **Estima la inclinación residual** de las coordenadas y agrupa sobre `y - b·x`.
   Es la red de seguridad del enderezado del paso 2: si la imagen ya vino
   derecha da ~0 y no cambia nada, y si el LLM no encontró la tabla (entonces no
   hubo recorte ni enderezado) rescata igual la mayor parte del problema.
2. **Deriva la tolerancia del paso entre renglones**, medido por autocorrelación,
   y no del alto de caja. El alto de caja no es la altura del renglón:
   `text_det_unclip_ratio` dilata las cajas, y en factura2 el alto mediano es de
   38 px contra un paso real de 27 px.

Con `verbose=True` imprime la inclinación, el paso y la tolerancia que eligió.


In [ ]:
filas = pipeline.agrupar_filas(res, verbose=True)
pipeline.imprimir_filas(filas)

In [ ]:
df = pipeline.filas_a_dataframe(filas)
df

## 5. Filas → items nombrados

Son dos preguntas distintas y se resuelven en dos lugares distintos:

| | |
|---|---|
| **cómo se llama** cada columna | `encabezado.py` — vocabulario, sin coordenadas |
| **dónde empieza** cada columna | `esquema.py` — geometría, sin vocabulario |

La grilla se descubre **de los datos**, no de los títulos: se proyectan todas las
cajas sobre el eje x contando en cuántas filas está ocupado cada píxel, y un
separador es un hueco que casi ninguna fila cruza. El encabezado sólo le pone
nombre a lo que ya se descubrió.

Usar la posición de los títulos como límite no funciona, y no es un detalle:
en factura5 `Descripción` está centrado sobre su columna y arranca 190 px a la
derecha de donde arrancan las descripciones, así que la descripción entera
terminaría en la columna de código. Y en factura4 hay una columna de código de
barras que directamente no tiene encabezado.


In [ ]:
doc, avisos = pipeline.estandarizar(filas)
df_items = pipeline.items_a_dataframe(doc)
df_items

### Qué hizo, exactamente

`estandarizar` no falla ruidosamente casi nunca: si algo salió mal, devuelve una
tabla plausible con una fila de menos o una columna corrida. Esta celda es donde
eso se ve.

Lo que hay que mirar, en orden:

- **grilla**: `datos (N columnas)` es lo normal. `encabezado` significa que no
  hubo suficientes filas para descubrir la grilla y se cayó al reparto por
  títulos, que es peor.
- **columnas sin título**: existen en los datos pero ningún encabezado las
  reclamó. Su contenido se pierde.
- **descartadas**: líneas que no parecieron items. `DESPACHO: S/FACT102141` y las
  de totales está bien que caigan acá; un producto de verdad, no.
- **celdas con dos valores**: en una columna atómica (código, cantidad, precio,
  importe) es el síntoma de dos filas que quedaron fusionadas en una.


In [ ]:
pipeline.imprimir_avisos(avisos)

### Validación: `cantidad × precio ≈ importe`

La única verificación que no necesita ground truth, y la red que atrapa una
columna bien mapeada con el número equivocado — que es lo que pasa cuando el OCR
lee `11128.28` donde dice `11128.46`.

`incompleta` no es un fallo: es una línea a la que le falta alguno de los tres
números, así que no hay nada que multiplicar.


In [ ]:
pipeline.validar_aritmetica(doc)

### Si el formato es nuevo *(opcional)*

Cuando la tabla de alias de `encabezado.py` no reconoce los títulos, hay un
segundo nivel: mandarle **sólo la fila de encabezados** a un LLM de texto. Son
~100 tokens y no dependen de la cantidad de items; los números nunca pasan por
el modelo, que es donde un modelo chico alucina caro y sin dejar rastro.

Está apagado por default a propósito: el camino de arriba es offline y
determinista. En los cinco formatos del corpus no se dispara nunca.


In [ ]:
# Hace una llamada de red. Sólo si `imprimir_avisos` mostró que faltan
# campos esenciales.
doc_llm, avisos_llm = pipeline.estandarizar(filas, usar_llm=True)
pipeline.imprimir_avisos(avisos_llm)
pipeline.items_a_dataframe(doc_llm)

---

## Antes de commitear este notebook

`Editar → Borrar todos los resultados`. Con los outputs del OCR embebidos el
archivo pesa 20x más y los diffs se vuelven ilegibles.
